In [21]:
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from SG_solver import rbf_matrix, second_divided_difference
from fractal_SG_solver_new import d2_fractal_L_W2 
from fractal_SG_solver_new import ddphi, H5_dd, pointwise_fractal
from alpha_fractal_function import alpha_fractalize,alpha_fractalize_second_derivative

In [ ]:
# Paper Example 1 parameters.
a = -1.0
b = 1.0
#######################
K = 8
s = 0.8
c = 0.027 
################
h = 0.01
tau = 0.01
################
T = 1

n = (b - a) / h

if n != int(n):
    raise ValueError("h must divide b-a exactly.")

n = int(n)

if n % K != 0:
    raise ValueError("n must be divisible by K because the paper defines N = n / K.")

N = n // K
Nt = int(round(T / tau))

x = np.linspace(a, b, n + 1)

# Interpolation center indices k_j, j = 1,...,N
k_idx = np.concatenate(([1], K * np.arange(1, N - 1), [n - 1])).astype(int)

xk = x[k_idx]
xkj = []
for j in range(1, N+1):
    if j == 1:
        value = float(x[1])
        xkj.append(value)

    elif 1 < j < N:
        value = a + ((j - 1) * (b - a)) / N
        xkj.append(value)
    elif j == N:
        value = float(x[n - 1])
        xkj.append(value)

In [ ]:
def f(x):
    y = np.sin(np.pi * x)
    return np.where(np.isclose(y, 0.0, atol=1e-12), 0.0, y)

def odd_dirichlet_extension(z):
    y = ((z + 1) % 4) - 1   

    if y <= 1:
        return pointwise_fractal(y, sine_pi_fractal)

    return -pointwise_fractal(2 - y, sine_pi_fractal)


f_beta = [ 0.005, 0.0025, 0.0025, 0.005]
n_subintervals = len(f_beta)
n_iter = 6
X = np.linspace(a, b, n_subintervals + 1)

f(-1) = 0.0, f(1) = 0.0
g(-1) = -0.0, g(1) = 0.0


In [ ]:
f_alpha = [0.0002,0.002,0.0003,0.0003,0.002,0.0002]
subintervals =len(f_alpha)
iter = 2
fractal_dd = alpha_fractalize_second_derivative(ddphi, H5_dd, -2, 2, subintervals, f_alpha, iter)

In [ ]:
# Build approximation of u_xx at t=0
A0 = rbf_matrix(xk, s)
cond_num = np.linalg.cond(A0)
print(f"Condition Number (2-norm): {cond_num:.4e}")


## Building fractalized numerical solution

In [ ]:
# Even base function
def g1(x):
    return (x**2 - x**4)/(np.exp(x) + 2)



E_sine_pi_fractal = alpha_fractalize(f, g1, -1, 1, n_subintervals, f_beta, n_iter)


def E_f_exact_u(x, t):
    return 0.5 * (
        odd_dirichlet_extension(x + t)
        + odd_dirichlet_extension(x - t)
    )

# Compute the exact solution at time T
E_f_u_exact = np.asarray([E_f_exact_u(xi, T) for xi in x])

# Initial condition at t=0
E_f_U = np.zeros((Nt + 2, len(x)))
E_f_U[0, :] = np.asanyarray([pointwise_fractal(x[d], E_sine_pi_fractal) for d in range(len(x))])  # initial condition at t=0

E_f_U[0, 0]  = 0.0
E_f_U[0, -1] = 0.0


E_rhs0 = []
for kj in k_idx:
    E_rhs0.append(second_divided_difference(x, E_f_U[0, :], kj))
E_rhs0 = np.asarray(E_rhs0)

E_alpha0 = np.linalg.solve(A0, E_rhs0)


E_f_uxx0 = np.zeros(len(x))
for i in range(len(x)):
    E_f_uxx0[i] = d2_fractal_L_W2(i, x, E_f_U[0, :], xk, E_alpha0, s, fractal_dd)

# Since g(x)=0
E_f_U[1, :] = E_f_U[0, :] + 0.5 * tau**2 * E_f_uxx0
E_f_U[1, 0]  = 0.0
E_f_U[1, -1] = 0.0

for d in tqdm(range(1, Nt + 1), desc="Time stepping", unit="step"):
    A = rbf_matrix(xk, s)
    rhs_d = np.asarray([second_divided_difference(x, E_f_U[d], kj) for kj in k_idx])
    alpha2 = np.linalg.solve(A, rhs_d)
    # alpha2 = A_inv @ rhs_d

    E_f_uxx = np.zeros(len(x))
    for i in range( len(x)):
        E_f_uxx[i] = d2_fractal_L_W2(i, x, E_f_U[d], xk, alpha2, s, fractal_dd)

    E_f_U[d + 1, :] = 2.0 * E_f_U[d, :] - E_f_U[d - 1, :] + tau**2 * E_f_uxx

    # Dirichlet boundary conditions
    E_f_U[d + 1, 0] = 0.0
    E_f_U[d + 1, -1] = 0.0

E_f_u_num = E_f_U[Nt, :]

E_f_err = E_f_u_num - E_f_u_exact
E_f_Linf = np.max(np.abs(E_f_err))
E_f_RMS = np.sqrt(np.sum(np.abs(E_f_err)**2)) / (n + 1)

print("f_Linf error for Even base funciton is", E_f_Linf)
print("f_RMS error for Even base funcitonis", E_f_RMS)


In [ ]:
# Odd base function
def g2(x):
    return x * (1 - x**2)

O_sine_pi_fractal = alpha_fractalize(f, g2, -1, 1, n_subintervals, f_beta, n_iter)


def O_f_exact_u(x, t):
    return 0.5 * (
        odd_dirichlet_extension(x + t)
        + odd_dirichlet_extension(x - t)
    )

# Compute the exact solution at time T
O_f_u_exact = np.asarray([O_f_exact_u(xi, T) for xi in x])

# Initial condition at t=0
O_f_U = np.zeros((Nt + 2, len(x)))
O_f_U[0, :] = np.asanyarray([pointwise_fractal(x[d], O_sine_pi_fractal) for d in range(len(x))])  # initial condition at t=0

O_f_U[0, 0]  = 0.0
O_f_U[0, -1] = 0.0


O_rhs0 = []
for kj in k_idx:
    O_rhs0.append(second_divided_difference(x, O_f_U[0, :], kj))
O_rhs0 = np.asarray(O_rhs0)

O_alpha0 = np.linalg.solve(A0, O_rhs0)


O_f_uxx0 = np.zeros(len(x))
for i in range(len(x)):
    O_f_uxx0[i] = d2_fractal_L_W2(i, x, O_f_U[0, :], xk, O_alpha0, s, fractal_dd)

# Since g(x)=0
O_f_U[1, :] = O_f_U[0, :] + 0.5 * tau**2 * O_f_uxx0
O_f_U[1, 0]  = 0.0
O_f_U[1, -1] = 0.0

for d in tqdm(range(1, Nt + 1), desc="Time stepping", unit="step"):
    A = rbf_matrix(xk, s)
    rhs_d = np.asarray([second_divided_difference(x, O_f_U[d], kj) for kj in k_idx])
    alpha2 = np.linalg.solve(A, rhs_d)
    # alpha2 = A_inv @ rhs_d

    O_f_uxx = np.zeros(len(x))
    for i in range( len(x)):
        O_f_uxx[i] = d2_fractal_L_W2(i, x, O_f_U[d], xk, alpha2, s, fractal_dd)

    O_f_U[d + 1, :] = 2.0 * O_f_U[d, :] - O_f_U[d - 1, :] + tau**2 * O_f_uxx

    # Dirichlet boundary conditions
    O_f_U[d + 1, 0] = 0.0
    O_f_U[d + 1, -1] = 0.0

O_f_u_num = O_f_U[Nt, :]

O_f_err = O_f_u_num - O_f_u_exact
O_f_Linf = np.max(np.abs(O_f_err))
O_f_RMS = np.sqrt(np.sum(np.abs(O_f_err)**2)) / (n + 1)

print("f_Linf error for Odd base funciton is", O_f_Linf)
print("f_RMS error for Odd base funcitonis", O_f_RMS)


In [ ]:
plt.figure(figsize=(10, 6))


plt.plot(
        x,
        E_f_err,
        label=f"Fractal Quasi-Interpolant Even Base Function",
        linewidth=1.5
    )
plt.plot(
        x,
        O_f_err,
        label=f"Fractal Quasi-Interpolant Odd Base Function",
        linewidth=1.5
    )

plt.xlabel("x")
plt.ylabel("Error")
plt.title(f"Pointwise Error at T = {T} for Different Base Funcitons")
plt.grid(True)
plt.legend()

plt.savefig(f"Images/pointwise_error_base_fucn.eps", format="eps")
plt.show()